# Fish to Fish Regression on All Species

## Import data and define columns

In [23]:

import pandas as pd
import numpy as np

from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (StandardScaler, OneHotEncoder)
from sklearn.impute import SimpleImputer
from sklearn.svm import SVR
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)


df = pd.read_csv(
    "../adore_dataset/processed/t-F2F_mortality.csv",
    dtype=str
)

#species columns 
df["Full_Species"] = (
    df["tax_genus"].astype(str)
    + " "
    + df["tax_species"].astype(str)
)

species_col = "Full_Species"

# features
target = "result_conc1_mean_mol_log"

categorical_cols = [
    "test_exposure_type",
    "result_conc1_type",
    "test_media_type"
]

ordinal_cols = ["result_obs_duration_mean"]

numeric_cols = [
    "chem_mw",
    "chem_ws",
    "chem_mp",
    "chem_rdkit_clogp"
]

# numeric columns
for col in numeric_cols + ordinal_cols + [target]:
    df[col] = pd.to_numeric(
        df[col], errors="coerce"
    )

# macc fingerprint
def parse_maccs(value):
    if not isinstance(value, str):
        return None

    value = value.strip()

    if len(value) != 167:
        return None

    if not set(value).issubset({"0", "1"}):
        return None

    return [int(bit) for bit in value]

valid_fp = df["chem_MACCS_fp"].apply(parse_maccs)

df = df.loc[valid_fp.notna()].copy()

maccs = pd.DataFrame(
    valid_fp.loc[df.index].tolist(),
    index=df.index,
    columns=[f"MACCS_{i:03d}" for i in range(167)]
)




## Splits and features

In [24]:
# X and y
X = pd.concat([
    df[categorical_cols + ordinal_cols + numeric_cols],
    maccs
], axis=1)

y = df[target]

# test splits
train = df["split_occurrence"] != "test"
test = df["split_occurrence"] == "test"

valid = y.notna() & df[species_col].notna()

train = train & y.notna()
test = test & valid

X_train = X.loc[train]
X_test = X.loc[test]

y_train = y.loc[train]
y_test = y.loc[test]

In [25]:
# maccs bits
fp_cols = [
    c for c in maccs.columns
    if X_train[c].std() >= 0.1
]

features = (
    categorical_cols
    + ordinal_cols
    + numeric_cols
    + fp_cols
)

X_train = X_train[features]
X_test = X_test[features]

## Preprocessing and Models

In [26]:
# preprocessing
numeric_features = (
    ordinal_cols + numeric_cols + fp_cols
)

preprocessing = ColumnTransformer([
    (
        "numeric",
        make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler()
        ),
        numeric_features
    ),
    (
        "categorical",
        make_pipeline(
            SimpleImputer(
                strategy="most_frequent"
            ),
            OneHotEncoder(
                handle_unknown="ignore"
            )
        ),
        categorical_cols
    )
])


In [27]:
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (RandomForestRegressor,GradientBoostingRegressor)
from sklearn.base import clone
from sklearn.metrics import (mean_absolute_error,root_mean_squared_error,r2_score)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Elastic Net": ElasticNet(),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    )
}



## Grid Search and Errors

In [28]:
# grid search
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline

# use predefined split to define folds
fold_ids = pd.to_numeric(df.loc[X_train.index, "split_occurrence"], errors="coerce")
assert fold_ids.notna().all(), "non-numeric fold ids — check split_occurrence values"
cv = PredefinedSplit(fold_ids.astype(int).values)

# define grids to search
param_grids = {
    "Ridge":       {"reg__alpha": [0.01, 0.1, 1, 10, 100]},
    "Lasso":       {"reg__alpha": [1e-4, 1e-3, 1e-2, 0.1]},
    "Elastic Net": {"reg__alpha": [1e-3, 1e-2, 0.1],
                    "reg__l1_ratio": [0.2, 0.5, 0.8]},
    "KNN":         {"reg__n_neighbors": [3, 5, 7, 11, 15],
                    "reg__weights": ["uniform", "distance"]},
    "SVR":         {"reg__C": [0.3, 1, 3, 10, 30],
                    "reg__epsilon": [0.05, 0.1, 0.2, 0.5],
                    "reg__gamma": ["scale", 0.01, 0.03]},
    "Decision Tree": {"reg__max_depth": [5, 10, 20, None],
                      "reg__min_samples_leaf": [1, 5, 10, 20]},
    "Random Forest": {"reg__n_estimators": [300],
                      "reg__max_depth": [10, 20, None],
                      "reg__max_features": ["sqrt", 0.3, 0.6],
                      "reg__min_samples_leaf": [1, 3]},
    "Gradient Boosting": {"reg__n_estimators": [200, 500],
                          "reg__learning_rate": [0.03, 0.1],
                          "reg__max_depth": [2, 3, 4],
                          "reg__subsample": [0.8, 1.0]},
    # Linear Regression has nothing to tune
}

# ---- run the searches ----
searches = {}
tuning_summary = []

for name, regressor in models.items():
    pipe = Pipeline([
        ("prep", clone(preprocessing)), 
        ("reg", clone(regressor))])
    grid = param_grids.get(name)

    if not grid:                       # e.g. Linear Regression
        searches[name] = pipe.fit(X_train, y_train)
        continue

    gs = GridSearchCV(
        pipe, grid,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
        refit=True,
        verbose=1,
    )
    gs.fit(X_train, y_train)
    searches[name] = gs.best_estimator_   # already refit on all training folds

    tuning_summary.append({
        "Model": name,
        "CV_RMSE": -gs.best_score_,
        "Best_Params": gs.best_params_,
    })
    print(f"{name:18s} CV RMSE {-gs.best_score_:.3f}  {gs.best_params_}")

tuning_summary = pd.DataFrame(tuning_summary).sort_values("CV_RMSE")

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Ridge              CV RMSE 1.150  {'reg__alpha': 100}
Fitting 5 folds for each of 4 candidates, totalling 20 fits


/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.386329e+02, tolerance: 4.741e+00
  model = cd_fast.enet_coordinate_descent(
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.219387e+01, tolerance: 4.423e+00
  model = cd_fast.enet_coordinate_descent(
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or

Lasso              CV RMSE 1.111  {'reg__alpha': 0.01}
Fitting 5 folds for each of 9 candidates, totalling 45 fits


/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.172558e+01, tolerance: 4.423e+00
  model = cd_fast.enet_coordinate_descent(
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.162373e+01, tolerance: 4.741e+00
  model = cd_fast.enet_coordinate_descent(
/Users/parisjalloh/Library/Python/3.12/lib/python/site-packages/sklearn/linear_model/_coordinate_descent.py:842: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or

Elastic Net        CV RMSE 1.105  {'reg__alpha': 0.1, 'reg__l1_ratio': 0.2}
Fitting 5 folds for each of 10 candidates, totalling 50 fits
KNN                CV RMSE 1.208  {'reg__n_neighbors': 15, 'reg__weights': 'distance'}
Fitting 5 folds for each of 60 candidates, totalling 300 fits


KeyboardInterrupt: 

In [15]:

from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

all_predictions = []

for name, model in searches.items():
    y_pred = model.predict(X_test)

    results = pd.DataFrame({
        "Model": name,
        "Species": df.loc[X_test.index, "Full_Species"].values,
        "Actual": np.asarray(y_test),
        "Predicted": y_pred
    })

    results["Absolute_Error"] = abs(
        results["Actual"] - results["Predicted"]
    )

    results["Squared_Error"] = (
        results["Actual"] - results["Predicted"]
    ) ** 2

    all_predictions.append(results)

all_errors = pd.concat(all_predictions, ignore_index=True)


## Rankings

In [ ]:

species_model_errors = (
    all_errors.groupby(["Species", "Model"])
    .agg(
        MAE=("Absolute_Error", "mean"),
        RMSE=("Squared_Error", lambda x: np.sqrt(x.mean())),
        Max_Error=("Absolute_Error", "max"),
        Count=("Absolute_Error", "size")
    )
    .reset_index()
)

# Sort by largest average error
species_model_errors = species_model_errors.sort_values(
    "MAE", ascending=False
)

display(species_model_errors)


,Species,Model,MAE,RMSE,Max_Error,Count
158,Gadus morhua,Linear Regression,2.757201,2.758106,2.827866,2
159,Gadus morhua,Ridge,2.711571,2.712492,2.782236,2
97,Cyprinella lutrensis,Lasso,2.417116,2.417152,2.432827,3
260,Monopterus albus,Elastic Net,2.373498,2.373712,2.412787,4
146,Fundulus grandis,KNN,2.333030,2.333030,2.333030,1
...,...,...,...,...,...,...
427,Silurus glanis,Lasso,0.079831,0.085508,0.112206,4
431,Solea solea,KNN,0.060932,0.060932,0.060932,1
190,Hypophthalmichthys nobilis,Elastic Net,0.043913,0.043913,0.043913,1
194,Hypophthalmichthys nobilis,Ridge,0.036608,0.036608,0.036608,1


In [21]:
# find best model
tuning_summary = pd.DataFrame(tuning_summary)

display(tuning_summary.sort_values("CV_RMSE"))

best_model_name = (
    tuning_summary
    .sort_values("CV_RMSE")
    .iloc[0]["Model"]
)

best_model = searches[best_model_name]

print("Best Model:", best_model_name)


,Model,CV_RMSE,Best_Params
2,Elastic Net,1.105387,"{'reg__alpha': 0.1, 'reg__l1_ratio': 0.2}"
1,Lasso,1.110757,{'reg__alpha': 0.01}
0,Ridge,1.150464,{'reg__alpha': 100}
3,KNN,1.207831,"{'reg__n_neighbors': 15, 'reg__weights': 'dist..."


Best Model: Elastic Net


In [ ]:
# elastic net with highest error

all_errors["Squared_Error"] = (
    all_errors["Actual"] - all_errors["Predicted"]
) ** 2

best_predictions = all_errors[
    all_errors["Model"] == best_model_name
]

best_species_errors = (
    best_predictions
    .groupby("Species")
    .agg(
        MAE=("Absolute_Error", "mean"),
        RMSE=("Squared_Error", lambda x: np.sqrt(x.mean())),
        Max_Error=("Absolute_Error", "max"),
        Count=("Absolute_Error", "size")
    )
    .sort_values("MAE", ascending=False)
)

display(best_species_errors)

,MAE,RMSE,Max_Error,Count
Species,,,,
Monopterus albus,3.161152,3.162396,3.277001,4
Leuresthes tenuis,2.677084,2.690271,3.125178,26
Hypophthalmichthys molitrix,2.670987,2.700159,3.451544,14
Fundulus grandis,2.665640,2.665640,2.665640,1
Lagodon rhomboides,2.591743,2.751588,3.423129,4
...,...,...,...,...
Oncorhynchus gorbuscha,0.209500,0.228495,0.295010,8
Solea solea,0.208903,0.208903,0.208903,1
Heterobranchus longifilis,0.194565,0.194565,0.194565,1


## Test with Best Model

In [21]:
from sklearn.metrics import mean_squared_error
# Initialize your best model
best_model = ElasticNet(random_state=42)

# Fit on training data only
best_model = Pipeline([
    ("preprocessing", preprocessing),
    ("model", ElasticNet(random_state=42))
])

# Predict on test data
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

# Evaluate
print("Test R²:", r2_score(y_test, y_pred))
print("Test MAE:", mean_absolute_error(y_test, y_pred))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))

Test R²: 0.2943744820542057
Test MAE: 1.1982854526480313
Test RMSE: 1.4752686954590837


In [22]:
print("Train R²:", best_model.score(X_train, y_train))
print("Test R²:", best_model.score(X_test, y_test))

Train R²: 0.3356106636380326
Test R²: 0.2943744820542057
